# Data Preprocessing
In this notebook we go through the following preprocessing steps of the data before applying it to baseline models, to achieve maximum performance.

Dataset: yt8m_text_clean_baseline.csv

Preprocessing:
- Lowercased text
- Removed URLs and punctuation
- Combined title + tags
- Removed videos with < 2 tokens
Purpose:
- Text-only baseline modeling (TF-IDF)


In [3]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from collections import Counter
import json

In [ ]:
df = pd.read_csv("../data/Youtube/yt8m_merged.csv")

In [5]:
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""

    s = s.lower()

    # remove urls and domain-like tokens
    s = re.sub(r"(https?://\S+|www\.\S+|\b\S+\.(com|net|org|io|ly|be|tv|co|me|gg|ru|de|uk|pl|es|fr|it|gr)\S*)", "", s, flags=re.IGNORECASE)
    s = re.sub(r"[^a-z0-9\s]", " ", s)   # keep alphanum
    s = re.sub(r"\s+", " ", s)           # normalize spaces
    return s.strip()


In [6]:
def tags_to_text(x):
    if not isinstance(x, str):
        return ""
    try:
        tags = json.loads(x)
        if isinstance(tags, list):
            return " ".join(tags)
    except Exception:
        pass
    return ""


In [7]:
def build_text_feature(df):
    title_clean = df["title"].apply(clean_text)
    tags_clean = df["tags_json"].apply(tags_to_text).apply(clean_text)

    df["text"] = (title_clean + " " + tags_clean).str.strip()
    return df


In [8]:
df = build_text_feature(df)

### Check to ensure everything is as expected

In [9]:
print("Empty text rows:", (df["text"] == "").sum())
print("Avg text length:", df["text"].str.split().apply(len).mean())
print("Max text length:", df["text"].str.split().apply(len).max())

df["text"].head(10)


Empty text rows: 39211
Avg text length: 19.066685329849538
Max text length: 191


0    the longest starfox assault match ever part 5 ...
1    marek pietruczuk przed mistrzostwami polski w ...
2    android 5 0 lollipop for moto g update moto g ...
3    jesus segado pasarela larios 2013 m laga pased...
4    framing a steel deck fraiming with metal by 51...
5    phurion halo 3 montage 2 amazing phurion halo3...
6    titanlar n sava clash of the titans trailer ti...
7    new ducati hypermotard 1100 gp great riding on...
8    slk 250 2013 mercedes escondido mercedes benz ...
9                                             goldfish
Name: text, dtype: object

#### There is a significant number of rows with empty text and certain rows contain too little text to contain semantic value.
Hence we drop rows with less than two tokens

In [ ]:
df["text_len"] = df["text"].str.split().apply(len)

print("Rows before:", len(df))
df_clean = df[df["text_len"] >= 2].reset_index(drop=True)
print("Rows after :", len(df_clean))

df_clean.to_csv("../data/Youtube/yt8m_text_clean_baseline.csv", index=False, encoding="utf-8")